[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shhommychon/KorNorm/blob/feature%2Fbetter-g2pK/.dev_phonology/build_phonology_lut.ipynb)

###### 환경설정

In [1]:
!pip install -qq g2pK pandas

## [g2pK/table.csv](https://github.com/Kyubyong/g2pK/blob/master/g2pk/table.csv) ➔ Python LUT 변환기

In [2]:
import os
import g2pk

# g2pK 내부 리소스 경로 탐색
g2pk_resource_path = os.path.join(os.path.dirname(g2pk.__file__), "table.csv")
if not os.path.exists(g2pk_resource_path): raise FileNotFoundError

### 상수 매핑 테이블 정의

- 파이썬 파일에 삽입될 '상수 변수명' 자체를 문자열로 매핑합니다.
- 어말/공백 조건은 `O_EOW`를 사용합니다.

In [3]:
# CSV 헤더(후속 초성 조건) -> 상수명
CSV_HEADER_TO_ONSET = {
    "( ?)ᄀ": "O_GIYEOK",
    "( ?)ᄁ": "O_SSANGGIYEOK",
    "( ?)ᄂ": "O_NIEUN",
    "( ?)ᄃ": "O_DIGEUT",
    "( ?)ᄄ": "O_SSANGDIGEUT",
    "( ?)ᄅ": "O_RIEUL",
    "( ?)ᄆ": "O_MIEUM",
    "( ?)ᄇ": "O_BIEUP",
    "( ?)ᄈ": "O_SSANGBIEUP",
    "( ?)ᄉ": "O_SIOT",
    "( ?)ᄊ": "O_SSANGSIOT",
    "( ?)ᄌ": "O_JIEUT",
    "( ?)ᄍ": "O_SSANGJIEUT",
    "( ?)ᄎ": "O_CHIEUT",
    "( ?)ᄏ": "O_KIEUK",
    "( ?)ᄐ": "O_TIEUT",
    "( ?)ᄑ": "O_PIEUP",
    "( ?)ᄒ": "O_HIEUT",
    "(\\W|$)": "O_EOW",
}

# CSV 셀 안의 자모 -> 상수명 (종성)
CSV_CHAR_TO_CODA = {
    'ᆨ': "C_GIYEOK",
    'ᆩ': "C_SSANGGIYEOK",
    'ᆪ': "C_GIYEOK_SIOT",
    'ᆫ': "C_NIEUN",
    'ᆬ': "C_NIEUN_JIEUT",
    'ᆭ': "C_NIEUN_HIEUT",
    'ᆮ': "C_DIGEUT",
    'ᆯ': "C_RIEUL",
    'ᆰ': "C_RIEUL_GIYEOK",
    'ᆱ': "C_RIEUL_MIEUM",
    'ᆲ': "C_RIEUL_BIEUP",
    'ᆳ': "C_RIEUL_SIOT",
    'ᆴ': "C_RIEUL_TIEUT",
    'ᆵ': "C_RIEUL_PIEUP",
    'ᆶ': "C_RIEUL_HIEUT",
    'ᆷ': "C_MIEUM",
    'ᆸ': "C_BIEUP",
    'ᆹ': "C_BIEUP_SIOT",
    'ᆺ': "C_SIOT",
    'ᆻ': "C_SSANGSIOT",
    'ᆼ': "C_IEUNG",
    'ᆽ': "C_JIEUT",
    'ᆾ': "C_CHIEUT",
    'ᆿ': "C_KIEUK",
    'ᇀ': "C_TIEUT",
    'ᇁ': "C_PIEUP",
    'ᇂ': "C_HIEUT",
}

# CSV 셀 안의 자모 -> 상수명 (초성)
CSV_CHAR_TO_ONSET = {
    'ᄀ': "O_GIYEOK",
    'ᄁ': "O_SSANGGIYEOK",
    'ᄂ': "O_NIEUN",
    'ᄃ': "O_DIGEUT",
    'ᄄ': "O_SSANGDIGEUT",
    'ᄅ': "O_RIEUL",
    'ᄆ': "O_MIEUM",
    'ᄇ': "O_BIEUP",
    'ᄈ': "O_SSANGBIEUP",
    'ᄉ': "O_SIOT",
    'ᄊ': "O_SSANGSIOT",
    'ᄋ': "O_IEUNG",
    'ᄌ': "O_JIEUT",
    'ᄍ': "O_SSANGJIEUT",
    'ᄎ': "O_CHIEUT",
    'ᄏ': "O_KIEUK",
    'ᄐ': "O_TIEUT",
    'ᄑ': "O_PIEUP",
    'ᄒ': "O_HIEUT",
}

### LUT 생성 로직

- CSV의 정규식을 파싱하여 `(C_변환종성, O_변환초성, \"규칙번호\")` 튜플 문자열로 가공합니다.

In [4]:
import pandas as pd

def parse_rule_to_code(csv_rule_str):
    """
    `ᆨ\\1ᄊ(23)` 형식을 파싱하여 `(C_GIYEOK, O_SSANGSIOT, "23")` 문자열로 리턴
    """
    if pd.isna(csv_rule_str) or not str(csv_rule_str).strip():
        return None

    # 정규식 분해: (종성)\1(초성)(규칙)
    match = re.match(r"^([\u11a8-\u11c2]*?)\\1([\u1100-\u1112]*?)(?:\(([\d/]*?)\))?$", str(csv_rule_str).strip())
    if not match:
        return f"({csv_rule_str})"

    coda_char = match.group(1)
    onset_char = match.group(2)
    rule_raw = match.group(3) or ''

    # 규칙 정규화
    if rule_raw:
        rule_clean = '|'.join([ f"{n}항" for n in rule_raw.split('/') ])
    else:
        rule_clean = "Kyubyong/g2pK"

    # 상수 변수명 획득
    c_var = CSV_CHAR_TO_CODA.get(coda_char, "C_NONE") if coda_char else "C_NONE"
    o_var = CSV_CHAR_TO_ONSET.get(onset_char, "''") if onset_char else "''"

    return f"({c_var}, {o_var}, \"{rule_clean}\")"

In [5]:
# 훈민정음 조음 위치 기반 정렬 순서 정의
sort_order = [
    # [1] 아음 (어금니소리)
    "GIYEOK", "KIEUK", "SSANGGIYEOK",
    "GIYEOK_SIOT",    # ㄱ(아음) + ㅅ(치음)

    # [2] 설음 (혓소리)
    "NIEUN",
    "DIGEUT", "TIEUT", "SSANGDIGEUT",
    "NIEUN_JIEUT",    # ㄴ(설음) + ㅈ(치음)
    "NIEUN_HIEUT",    # ㄴ(설음) + ㅎ(후음)

    # [3] 순음 (입술소리)
    "MIEUM",
    "BIEUP", "PIEUP", "SSANGBIEUP",
    "BIEUP_SIOT",     # ㅂ(순음) + ㅅ(치음)

    # [4] 치음 (잇소리)
    "SIOT", "SSANGSIOT",
    "JIEUT", "CHIEUT", "SSANGJIEUT",

    # [5] 후음 (목구멍소리)
    "IEUNG", "HIEUT",

    # [6] 반설음 (ㄹ 계열)
    "RIEUL",
    "RIEUL_GIYEOK",   # + 아음(ㄱ)
    "RIEUL_TIEUT",    # + 설음(ㅌ)
    "RIEUL_MIEUM",    # + 순음(ㅁ)
    "RIEUL_BIEUP",    # + 순음(ㅂ)
    "RIEUL_PIEUP",    # + 순음(ㅍ)
    "RIEUL_SIOT",     # + 치음(ㅅ)
    "RIEUL_HIEUT",    # + 후음(ㅎ)

    # [7] 기타 특수 처리
    "NONE", "EOW"
]

def get_sort_key(key_string):
    """C_ 또는 O_ 접두사를 떼고 정렬 인덱스를 반환"""
    # 따옴표가 씌워져 있을 수 있으니 안전하게 제거
    clean_key = key_string.replace('"', '').replace('\'', '')
    base_name = clean_key.split('_', 1)[-1]
    try:
        return sort_order.index(base_name)
    except ValueError:
        return 999  # 목록에 없는 키는 맨 뒤로

In [6]:
import pandas as pd
import re

df = pd.read_csv(g2pk_resource_path)

# CSV 데이터를 중간 딕셔너리에 수집
lut_data = {}

for _, row in df.iterrows():
    coda_key_char = str(row.iloc[0]).strip()
    if not coda_key_char or pd.isna(row.iloc[0]): continue

    coda_key_var = CSV_CHAR_TO_CODA.get(coda_key_char, "C_NONE")
    if coda_key_var not in lut_data:
        lut_data[coda_key_var] = {}

    for col_name in df.columns[1:]:
        onset_key_var = CSV_HEADER_TO_ONSET.get(col_name.strip())
        if not onset_key_var: continue

        parsed_code = parse_rule_to_code(row[col_name])
        if parsed_code: # 규칙이 있을 때만 딕셔너리에 저장
            lut_data[coda_key_var][onset_key_var] = parsed_code

# 수집된 데이터를 조음 위치 순서로 정렬하여 텍스트 생성
lines = ["# 훈민정음 조음 위치 기반으로 정렬된 g2pK 정규식 데이터\n", "PHONOLOGY_LUT = {"]

# 종성(Coda) 정렬
sorted_codas = sorted(lut_data.keys(), key=get_sort_key)
for coda in sorted_codas:
    lines.append(f"    {coda}: {{")

    # 초성(Onset) 정렬
    sorted_onsets = sorted(lut_data[coda].keys(), key=get_sort_key)
    for onset in sorted_onsets:
        parsed_code = lut_data[coda][onset]
        lines.append(f"        {onset}: {parsed_code},")

    lines.append("    },")

lines.append("}")

In [7]:
for line in lines:
    print(line)

with open("apply_lut.py", 'w', encoding="utf-8") as f:
    f.write('\n'.join(lines))

# 훈민정음 조음 위치 기반으로 정렬된 g2pK 정규식 데이터

PHONOLOGY_LUT = {
    C_GIYEOK: {
        O_GIYEOK: (C_GIYEOK, O_SSANGGIYEOK, "23항"),
        O_NIEUN: (C_IEUNG, O_NIEUN, "18항"),
        O_DIGEUT: (C_GIYEOK, O_SSANGDIGEUT, "23항"),
        O_MIEUM: (C_IEUNG, O_MIEUM, "18항"),
        O_BIEUP: (C_GIYEOK, O_SSANGBIEUP, "23항"),
        O_SIOT: (C_GIYEOK, O_SSANGSIOT, "23항"),
        O_JIEUT: (C_GIYEOK, O_SSANGJIEUT, "23항"),
        O_HIEUT: (C_NONE, O_KIEUK, "12항"),
        O_RIEUL: (C_IEUNG, O_NIEUN, "19항|18항"),
    },
    C_KIEUK: {
        O_GIYEOK: (C_GIYEOK, O_SSANGGIYEOK, "9항|23항"),
        O_KIEUK: (C_GIYEOK, O_KIEUK, "9항"),
        O_SSANGGIYEOK: (C_GIYEOK, O_SSANGGIYEOK, "9항"),
        O_NIEUN: (C_IEUNG, O_NIEUN, "18항"),
        O_DIGEUT: (C_GIYEOK, O_SSANGDIGEUT, "9항|23항"),
        O_TIEUT: (C_GIYEOK, O_TIEUT, "9항"),
        O_SSANGDIGEUT: (C_GIYEOK, O_SSANGDIGEUT, "9항"),
        O_MIEUM: (C_IEUNG, O_MIEUM, "9항|18항"),
        O_BIEUP: (C_GIYEOK, O_SSANGBIEUP, "9항|23항"),
        O_PIEUP: (C_GIY